# Practica 3 — Procesamiento Digital de Imagenes

**Alumno:** Fernando Leon Franco
**Fecha de entrega:** _por definir_

---

## Resumen

_TODO: describir a grandes rasgos el objetivo y las tecnicas utilizadas en el desarrollo de la practica._

Esta practica aplica filtros basados en mascara de convolucion para resolver problemas de ruido y mejora de imagenes. Las tecnicas a usar son: filtrado pasa-bajas iterado, filtros para eliminacion de ruido (mediana, gaussiano, etc.), filtro de mediana sobre ruido uniforme, High-Boost combinado con ajustes de contraste y ecualizacion, deteccion de bordes, y reduccion de ruido por resta y promedio de imagenes consecutivas.

In [6]:
%matplotlib inline
import sys
from pathlib import Path
import matplotlib.pyplot as plt

BASE_DIR = Path().resolve().parents[1]
sys.path.insert(0, str(BASE_DIR))

from image_processing import DerivativeVisionNode

DATA_DIR = BASE_DIR / "image_processing" / "data"

files = {
    # E1, E3 — imagen base para pruebas
    "golf":         "golf.BMP",
    # E2 — eliminacion de ruido (estan en data/ruido/)
    "torre_verona": "ruido/torre_verona.bmp",
    "florencia":    "ruido/florencia.bmp",
    # PDF pide "florencia1.bmp" pero el archivo real del profe es florencia2.bmp.
    "florencia1":   "ruido/florencia2.bmp",
    # E4 — radiografia para High-Boost
    "torax":        "toraxP.bmp",
    # E5 — deteccion de bordes
    "waldo":        "waldo.bmp",
    # E6, E7 — fibra optica (la lista de 11 se genera con load_fibras)
}

def load_image(nombre: str) -> DerivativeVisionNode:
    return DerivativeVisionNode.desde_archivo(DATA_DIR / files[nombre])

# Las fibras se llaman "fibra3 (N).BMP" (con espacio y parentesis) en data/fibra/.
def load_fibras(n: int = 11) -> list[DerivativeVisionNode]:
    return [
        DerivativeVisionNode.desde_archivo(DATA_DIR / "fibra" / f"fibra3 ({i}).BMP")
        for i in range(1, n + 1)
    ]

## 1️⃣ — Ejercicio 1

Aplique el filtrado pasa-bajas varias veces sobre la misma imagen. Explique el resultado limite de aplicar este filtro un numero infinito de veces.

In [ ]:
# ──────────────────────────────────────────────────────────────
# E1 — Filtrado pasa-bajas iterado
# PDF: "Aplique el filtrado pasa-bajas varias veces sobre la misma
#       imagen. Explique el resultado limite de aplicar este filtro
#       un numero infinito de veces."
# Rodrigo: golf.bmp en color, gaussiano(size=5, sigma=1.0) × 7 iteraciones.
# ──────────────────────────────────────────────────────────────
imagen = load_image("golf")
imagen.mostrar(block=False)
imagen.histograma(block=False)

# Approach: aplicar el mismo pasa-bajas N veces y observar la tendencia.
# Pasa-bajas equivalentes en la lib (cualquiera sirve, gaussiano es lo clasico):
#   imagen.gaussiano(size=5, sigma=1.0)
#   imagen.suavizar(size=5)        # box filter
#   imagen.piramidal(size=5)        # Chebyshev
#
# Teoria del limite: cada convolucion equivale a un promedio local. Tras N
# pasadas, el "radio efectivo" crece como sqrt(N). En el infinito, cada
# pixel tiende al promedio GLOBAL → imagen plana de un solo tono (gris medio).
# Validacion cuantitativa: imagen.max() - imagen.min() → 0 conforme N crece.
#
# TODO: implementar el loop iterativo. Estructura sugerida:
#   actual = imagen
#   iteraciones_a_mostrar = (1, 5, 20, 50, 100)
#   for n in range(1, max(iteraciones_a_mostrar) + 1):
#       actual = actual.gaussiano(size=5, sigma=1.0)
#       if n in iteraciones_a_mostrar:
#           actual.title = f"N={n}"
#           actual.mostrar(block=False)
#           # opcional: actual.histograma(block=False)
#           print(f"N={n:3d}  rango=[{actual.min():.3f}, {actual.max():.3f}]")
#
# TODO: comparar el rango max-min entre iteraciones (debe colapsar).
# TODO: registrar a partir de que N el contenido ya no es reconocible.

plt.show()

### Explicaciones del Ejercicio 1

**Descripción:** _TODO — describir brevemente qué se hizo: imagen, filtro elegido (gaussiano/box/piramidal), tamaño y σ usados, número de iteraciones probadas._

**¿Cuál es el resultado límite de aplicar el filtro pasa-bajas un número infinito de veces?**

_TODO_

Pista para la respuesta: cada iteración pondera cada pixel con sus vecinos. El soporte efectivo del kernel acumulado crece con cada pasada (≈ σ·√N para un gaussiano). Cuando ese soporte supera el tamaño de la imagen, todos los pixeles dependen de todos — colapsan al promedio global. Visualmente: la imagen tiende a un **tono uniforme**, igual al promedio de la original. Cuantitativamente: `max - min → 0` y la varianza del histograma → 0.

## 2️⃣ — Ejercicio 2

Elimine el ruido presente en las fotografias **torre_verona.bmp**, **florencia.bmp** y **florencia1.bmp**. Describa de forma cualitativa y cuantitativa el tipo de ruido presente en la imagen. Justifique el filtro utilizado, asi como los valores de tamaño de ventana, umbral, etc.

In [ ]:
# ──────────────────────────────────────────────────────────────
# E2 — Eliminar ruido en torre_verona, florencia y florencia1
# PDF: "Elimine el ruido presente en las fotografias torre_verona.bmp,
#       florencia.bmp y florencia1.bmp. Describa de forma cualitativa y
#       cuantitativa el tipo de ruido. Justifique el filtro utilizado,
#       asi como los valores de tamaño de ventana, umbral, etc."
# Rodrigo:
#   - torre_verona  → filtro_mediana(radio=4) + filtro_umbral(40)
#   - florencia     → filtro_mediana(radio=1)
#   - florencia2    → filtro_mediana(radio=3) + filtro_umbral(40)
# (radio del amigo es en pixeles → size = 2·radio + 1; size=9, 3, 7)
# ──────────────────────────────────────────────────────────────
imagen_torre      = load_image("torre_verona")
imagen_florencia  = load_image("florencia")
imagen_florencia1 = load_image("florencia1")

for img in (imagen_torre, imagen_florencia, imagen_florencia1):
    img.mostrar(block=False)
    img.histograma(block=False)

# Approach por imagen:
#   1) Identificar el tipo de ruido viendo histograma + senal_por_canal()
#      (slider para recorrer filas).
#      - sal y pimienta:  picos extremos (0 y 255), pixeles aislados.
#      - gaussiano:       histograma "ensanchado" alrededor de los picos del original.
#      - uniforme:        ensanchamiento mas plano que el gaussiano.
#   2) Aplicar el filtro adecuado:
#      - sal y pimienta  → img.mediana(size) o img.mediana_cruz(size)
#      - gaussiano/uniforme → img.gaussiano(size, sigma) o img.filtro_sigma(size, sigma)
#   3) (opcional, recomendado) preservar detalle con filtro_umbral:
#      filtrada = img.mediana(size=9)
#      limpia   = img.filtro_umbral(filtrada, umbral=0.15)
#      Solo reemplaza pixeles donde la diferencia |orig - filtrada| > umbral
#      → suaviza zonas ruidosas pero respeta bordes y textura fina.
#
# Parametros sugeridos (a partir de Rodrigo, traducidos a tu API):
#   torre_verona      → mediana(size=9)  + filtro_umbral(0.15)
#   florencia         → mediana(size=3)
#   florencia1 (=2)   → mediana(size=7)  + filtro_umbral(0.15)
#
# TODO: para cada imagen aplicar el filtro elegido y mostrar:
#   limpia = img.mediana(size=...)
#   limpia.mostrar(block=False)
#   limpia.histograma(block=False)
#   img.mostrar_diferencias(limpia, magnifier=5.0, block=False)  # ver donde actuó
#
# TODO (cuantitativo): medir el cambio en el histograma. P.ej. usar
#   abs(img.tensor - limpia.tensor).mean() para ver cuanto se modifico.
#
# TODO: redactar para cada imagen: tipo de ruido + filtro + por que.

plt.show()

### Explicaciones del Ejercicio 2

**Descripción:** _TODO — qué imágenes se cargaron, qué se observó en el histograma original, qué filtro y parámetros se aplicaron a cada una._

**Tipo de ruido (cualitativo y cuantitativo) presente en cada imagen:**

- `torre_verona.bmp`: _TODO_ — pista: revisar histograma; ¿hay picos en 0 y 255 (sal y pimienta) o ensanchamiento general (gaussiano)?
- `florencia.bmp`: _TODO_
- `florencia1.bmp` (archivo real `florencia2.bmp`): _TODO_

**Justificación del filtro y de los parámetros (ventana, umbral, etc.):**

_TODO_

Pistas:
- **Mediana** es la mejor contra **sal y pimienta** porque los valores extremos quedan fuera del valor mediano; tamaño impar (size=3, 5, 7, 9) según qué tan denso esté el ruido.
- **Gaussiano** o **filtro sigma** funcionan mejor contra ruido aditivo continuo (gaussiano/uniforme); el sigma preserva bordes mejor que un gaussiano puro.
- **`filtro_umbral`** sirve para mezclar la versión filtrada con la original solo donde la diferencia sea grande — preserva detalle fino donde el filtro liso lo destruiría.

## 3️⃣ — Ejercicio 3

Utilizando un generador de ruido uniforme, pruebe el filtro de mediana con distintos valores de radio y porcentajes de ruido, en tonos de grises y en RGB. En base a sus pruebas ¿es posible establecer una relacion entre porcentaje de ruido y radio de mascara de filtrado? Explique.

In [ ]:
# ──────────────────────────────────────────────────────────────
# E3 — Ruido uniforme + mediana, barrido grises y RGB
# PDF: "Utilizando un generador de ruido uniforme, pruebe el filtro
#       de mediana con distintos valores de radio y porcentajes de
#       ruido, en tonos de grises y en RGB. ¿Es posible establecer
#       una relacion entre porcentaje de ruido y radio de mascara?"
# Rodrigo: barrido (0.05, 0.01, 0.10) × mediana radio (1, 2, 3),
#          en color y grises sobre golf.bmp.
# Nota: Rodrigo usa "ruido_porcentaje" (mete pixeles con valor random),
#       el PDF pide ruido UNIFORME aditivo (lo que da nuestra ruido_uniforme).
# ──────────────────────────────────────────────────────────────
imagen_color  = load_image("golf")
imagen_grises = imagen_color.escala_grises()
imagen_color.mostrar(block=False)
imagen_grises.mostrar(block=False)

# Approach: barrido sistematico amplitud × size_mediana, en color y grises.
#   amplitud:  0.05 ≈ ±13/255   (ruido suave)
#              0.10 ≈ ±25/255   (ruido medio, lo mas tipico)
#              0.20 ≈ ±51/255   (ruido fuerte)
#   size_mediana:  3, 5, 7      (ventana, debe ser impar)
#
# Estructura sugerida (loop doble):
#
#   amplitudes = (0.05, 0.10, 0.20)
#   sizes      = (3, 5, 7)
#   for base, etiqueta in [(imagen_grises, "grises"), (imagen_color, "color")]:
#       for amp in amplitudes:
#           ruidosa = base.ruido_uniforme(amplitud=amp, seed=42)
#           ruidosa.title = f"ruido ±{amp:.2f} ({etiqueta})"
#           ruidosa.mostrar(block=False)
#           for s in sizes:
#               filtrada = ruidosa.mediana(size=s)
#               filtrada.title = f"mediana {s} sobre ±{amp} ({etiqueta})"
#               filtrada.mostrar(block=False)
#               # opcional: filtrada.histograma(block=False)
#
# TODO: ejecutar el barrido (ojo: son 18 imagenes; quiza reducir a 2x2 si es mucho).
# TODO: opcionalmente probar mediana_cruz(size) — preserva mejor diagonales.
# TODO: medir cuantitativamente la calidad de cada filtrado, p.ej.:
#   error = (base.tensor - filtrada.tensor).abs().mean().item()
#   asi se ve si la mediana acerco la imagen al original.
#
# Conclusion esperada: a mas amplitud de ruido, mas size del filtro hace falta,
# pero pasarse de size pierde detalle (la mediana suaviza tambien la señal real).
# Hay un "sweet spot" para cada nivel de ruido.

plt.show()

### Explicaciones del Ejercicio 3

**Descripción:** _TODO — qué amplitudes de ruido se probaron, qué tamaños de mediana, en grises y en RGB._

**¿Es posible establecer una relación entre porcentaje de ruido y radio de máscara de filtrado?**

_TODO_

Pista: a mayor amplitud (o porcentaje) de ruido, **se necesita mayor tamaño de ventana** para que la mediana logre descartarlo, pero **a costa de perder detalle**. Para ruido suave (`amplitud=0.05`) basta `size=3`; para ruido fuerte (`amplitud=0.20`) hace falta `size=7` o más. El criterio práctico: la ventana debe ser lo suficientemente grande para que la mayoría de los píxeles dentro NO estén afectados por el ruido — así el valor mediano captura la señal real.

**Diferencias entre procesamiento en grises y en RGB:**

_TODO_

Pista: en RGB la mediana se aplica canal por canal independientemente. Si el ruido afecta los 3 canales con diferente intensidad, puede aparecer **moteado de color** (artefactos cromáticos) que no se ve en grises. La mediana en grises es más limpia visualmente.

## 4️⃣ — Ejercicio 4

Utilizando la tecnica **High-Boost**, en conjunto con los ajustes de contraste y ecualizacion probados anteriormente, busque un mejor resultado para el tratamiento de la radiografia. Pruebe ajustar primero el contraste seguido del filtrado, asi como filtrado primero y terminar con ajuste de contraste. ¿Hay alguna diferencia en el orden?

In [ ]:
# ──────────────────────────────────────────────────────────────
# E4 — High-Boost + ajustes de contraste/ecualizacion en radiografia
# PDF: "Utilizando la tecnica 'High-Boost', en conjunto con los ajustes
#       de contraste y ecualizacion probados anteriormente, busque un
#       mejor resultado para el tratamiento de la radiografia. Pruebe
#       ajustar primero el contraste seguido del filtrado, asi como
#       filtrado primero y terminar con ajuste de contraste. ¿Hay alguna
#       diferencia en el orden?"
# Rodrigo: ecualizar() → highboost(laplaciano, factor=3.0)  Y  al reves.
# Tu lib: receta manual (la que ya usaste en proyecto_costal.py):
#         (img - lap_crudo * k).clip()
#         con lap_crudo = img.laplaciano(extendido=True, crudo=True)
# ──────────────────────────────────────────────────────────────
imagen_torax = load_image("torax").escala_grises()
imagen_torax.title = "Radiografia original"
imagen_torax.mostrar(block=False)
imagen_torax.histograma(block=False)

# Approach: comparar dos ordenes con el MISMO ajuste de contraste y la MISMA k.
# Sugerido: k = 1.0 a 2.0 (en escala 0-1; equivalente a factor=3 que usa Rodrigo
# en escala 0-255, ya que su laplaciano no es "crudo" sino normalizado por filter2D).
# Recordar: el resultado se llama "high-boost" porque eleva las frecuencias altas
# (los bordes) sumando una version del laplaciano negada.
#
# Receta high-boost en tu lib:
#   def high_boost(img, k):
#       lap = img.laplaciano(extendido=True, crudo=True)   # crudo=True preserva signo
#       return (img - lap * k).clip()                       # clip a [0,1] al final
#
# ── ORDEN A: contraste → high-boost ────────────────────
# TODO:
#   ajustada_A = imagen_torax.ecualizar()
#   # Alternativas: imagen_torax.transformacion_gamma(2.0)
#   #               imagen_torax.transformacion_lineal(entrada=(0,40,70,255), salida=(0,140,180,255))
#   ajustada_A.title = "A: ecualizada"
#   ajustada_A.mostrar(block=False)
#
#   k = 1.5
#   lap_A = ajustada_A.laplaciano(extendido=True, crudo=True)
#   resultado_A = (ajustada_A - lap_A * k).clip()
#   resultado_A.title = f"A: ecualizar -> high-boost (k={k})"
#   resultado_A.mostrar(block=False)
#   resultado_A.histograma(block=False)
#
# ── ORDEN B: high-boost → contraste ────────────────────
# TODO:
#   k = 1.5
#   lap_B = imagen_torax.laplaciano(extendido=True, crudo=True)
#   boost_B = (imagen_torax - lap_B * k).clip()
#   boost_B.title = f"B: high-boost (k={k})"
#   boost_B.mostrar(block=False)
#
#   resultado_B = boost_B.ecualizar()    # mismo ajuste que en A
#   resultado_B.title = "B: high-boost -> ecualizar"
#   resultado_B.mostrar(block=False)
#   resultado_B.histograma(block=False)
#
# TODO: comparar visualmente A vs B (observar bordes pulmon/cadera).
# TODO: usar resultado_A.mostrar_diferencias(resultado_B, magnifier=3.0) para ver
#       donde difieren los dos pipelines.

plt.show()

### Explicaciones del Ejercicio 4

**Descripción:** _TODO — qué ajuste de contraste se eligió (ecualizar / gamma / lineal por tramos), valor de k usado en high-boost, y los dos pipelines (A y B)._

**¿Hay alguna diferencia en el orden (contraste → filtrado vs filtrado → contraste)?**

_TODO_

Pista: **sí cambia**, y bastante. Razones:
- **Orden A (contraste → high-boost):** el ajuste de contraste estira el rango de intensidades primero, luego el laplaciano detecta bordes sobre la imagen ya estirada → los bordes resultan **más fuertes** porque tienen más rango disponible para amplificar. Posible problema: si el contraste sobre-satura, el laplaciano "ve" bordes falsos en zonas saturadas.
- **Orden B (high-boost → contraste):** primero se enfatizan los bordes, luego el ajuste de contraste **redistribuye** todo el rango — incluyendo los picos generados por el high-boost. Resultado: contraste global más uniforme pero los bordes pueden quedar **menos pronunciados** porque la ecualización los redistribuye junto con el resto.

En general el **orden A** suele dar imágenes más nítidas para diagnóstico médico (bordes resaltados explícitamente), mientras el **orden B** da imágenes más balanceadas globalmente.

## 5️⃣ — Ejercicio 5

Elija una fotografia para aplicar la tecnica de deteccion de bordes. El resultado del procesamiento debe parecer el trazado en blanco y negro del perfil de los objetos.

In [ ]:
# ──────────────────────────────────────────────────────────────
# E5 — Deteccion de bordes (trazado en B/N)
# PDF: "Elija una fotografia para aplicar la tecnica de deteccion de
#       bordes. El resultado del procesamiento debe parecer el trazado
#       en blanco y negro del perfil de los objetos."
# Rodrigo: waldo.bmp en color con mostrar_bordes(laplaciano).
# Importante del PDF: salida B/N (no color).
# ──────────────────────────────────────────────────────────────
imagen = load_image("waldo").escala_grises()
imagen.title = "Original (grises)"
imagen.mostrar(block=False)
imagen.histograma(block=False)

# Approach: 1) suavizar suave para reducir ruido (LoG), 2) laplaciano,
# 3) post-proceso para que parezca un "trazado".
#
# La lib tiene laplaciano con dos variantes:
#   - extendido=False → kernel 4 vecinos (clasico, mas direccional)
#   - extendido=True  → kernel 8 vecinos (incluye diagonales, mas isotropico) ← preferido
#
# TODO (basico — laplaciano directo sobre original):
#   bordes = imagen.laplaciano(extendido=True)
#   bordes.title = "Bordes (laplaciano 8 vecinos)"
#   bordes.mostrar(block=False)
#
# TODO (mejor — LoG = Laplacian of Gaussian: suavizar antes reduce ruido):
#   suave   = imagen.gaussiano(size=5, sigma=1.0)
#   bordes_log = suave.laplaciano(extendido=True)
#   bordes_log.title = "LoG (gaussiano + laplaciano)"
#   bordes_log.mostrar(block=False)
#
# TODO (estilizar el resultado a "trazado puro" en B/N):
#   bordes_realzados = bordes_log.estirar_contraste()      # min-max a [0,1]
#   trazado          = bordes_realzados.binarizar(0.15)     # umbral global
#   trazado.title    = "Trazado B/N"
#   trazado.mostrar(block=False)
#   # Si queda muy "gordo": aumentar umbral; si queda muy fino: bajarlo.
#
# TODO (opcional — invertir si quedan lineas blancas sobre fondo negro y
# preferis lineas negras sobre fondo blanco):
#   trazado.negativo().mostrar(block=False)

plt.show()

### Explicaciones del Ejercicio 5

**Descripción:** _TODO — qué imagen se eligió, qué operador (Laplaciano 4/8 vecinos, LoG), umbral de binarización si se aplicó._

**Operador elegido y por qué:**

_TODO_

Pistas:
- El **Laplaciano** es la segunda derivada espacial (∇²f). Detecta bordes como cruces por cero o picos de magnitud. Es el operador disponible en tu lib y cubre el ejercicio.
- **`extendido=True`** (8 vecinos) es más isotrópico que `extendido=False` (4 vecinos): captura mejor los bordes diagonales.
- **LoG** (Laplacian of Gaussian: suavizar primero con gaussiano, luego laplaciano) reduce el ruido en los bordes — el laplaciano puro amplifica cualquier ruido alta-frecuencia.
- Para el "trazado en B/N" del PDF: combinar `estirar_contraste()` + `binarizar(umbral)` para forzar solo dos tonos (negro y blanco).

## 6️⃣ — Ejercicio 6

En la carpeta **FIBRA** hay multiples imagenes de una fibra optica dopada con Erbio cuyo nucleo brilla en color verde cuando se bombea con un diodo laser de 1500 nm. Estas imagenes fueron tomadas en condiciones de baja iluminacion y con una camara que era particularmente ruidosa. Realice la **resta entre dos imagenes consecutivas** para evaluar el ruido. Puede usar el histograma para revisar si el ruido tiene una distribucion normal.

In [ ]:
# ──────────────────────────────────────────────────────────────
# E6 — Resta entre dos fibras consecutivas para evaluar ruido
# PDF: "En la carpeta FIBRA hay multiples imagenes de una fibra optica
#       dopada con Erbio cuyo nucleo brilla en color verde cuando se
#       bombea con un diodo laser de 1500 nm. Estas imagenes fueron
#       tomadas en condiciones de baja iluminacion y con una camara
#       que era particularmente ruidosa. Realice la resta entre dos
#       imagenes consecutivas para evaluar el ruido. Puede usar el
#       histograma para revisar si el ruido tiene una distribucion normal."
# Rodrigo: fibra3 (2) - fibra3 (3) y al reves.
# ──────────────────────────────────────────────────────────────
fibras = load_fibras(11)
fibra_a = fibras[0].escala_grises()
fibra_b = fibras[1].escala_grises()
fibra_a.title = "fibra3 (1)"
fibra_b.title = "fibra3 (2)"
fibra_a.mostrar(block=False)
fibra_b.mostrar(block=False)
fibra_a.histograma(block=False)

# Approach: la senal real (la fibra) es practicamente identica entre tomas
# consecutivas; lo que cambia es el ruido aleatorio del sensor. Por eso la
# resta cancela la senal y deja ~puro ruido.
#
# OJO con el clip: la resta puede dar valores negativos. El operador `-` de
# tu lib NO clampea (estilo MathCad). Para visualizar bien hay dos opciones:
#
# Opcion A (cruda, lo que pide el PDF para histograma):
#   residuo = fibra_a - fibra_b   # tensor con valores en [-1, +1] aprox
#   # No mostrar() directo (clipea a [0,1]); en su lugar:
#   residuo_clipeado = residuo.clip()                  # informacion incompleta
#   residuo_clipeado.histograma(block=False)
#
# Opcion B (mejor para ver la campana del histograma):
#   residuo_centrado = (fibra_a - fibra_b + 0.5).clip()  # offset visual a 0.5
#   residuo_centrado.title = "residuo + 0.5 (visual)"
#   residuo_centrado.mostrar(block=False)
#   residuo_centrado.histograma(block=False)
#
# TODO: aplicar Opcion B y observar la pinta del histograma:
#   - Si es ruido normal/gaussiano → campana centrada en 0.5, simetrica.
#   - Si tiene cola larga o asimetria → algo no-gaussiano (ruido tipo Poisson, o sesgo del sensor).
#
# TODO (opcional, cuantitativo): calcular media y desviacion estandar del residuo:
#   residuo_t = (fibra_a.tensor - fibra_b.tensor)
#   print(f"media={residuo_t.mean():.4f}, std={residuo_t.std():.4f}")
#   Una σ pequeña (≈0.01-0.05 en escala 0-1) indica ruido suave.

plt.show()

### Explicaciones del Ejercicio 6

**Descripción:** _TODO — qué dos fibras se restaron, qué se observó en el residuo (media, σ, forma del histograma)._

**¿El ruido tiene distribución normal? (justificar con histograma):**

_TODO_

Pista: la idea de la resta entre tomas consecutivas es que **la señal real se cancela** (la fibra no se mueve entre frames) y queda solo la diferencia de ruido. Si cada toma tiene ruido `r_i ~ N(0, σ²)` independiente, el residuo `r_1 - r_2` también es normal pero con varianza doble: `~ N(0, 2σ²)`. En el histograma del residuo (centrado con `+0.5`):
- **Forma de campana simétrica** alrededor de 0.5 → ruido gaussiano. ✓
- **Asimetría o colas largas** → ruido no-gaussiano (tipo Poisson en baja luz, o ruido por defectos del sensor).

Cuantitativamente: si `media ≈ 0` (no hay sesgo entre tomas) y `std` es pequeña, el ruido es suave; si `std` es grande, la cámara es muy ruidosa.

## 7️⃣ — Ejercicio 7

Otra forma de reducir el ruido sin atenuar las altas frecuencias es **promediar imagenes consecutivas**. Esta tecnica se usa frecuentemente en Astronomia. Promedie 11 imagenes consecutivas de la fibra y compare el resultado con la imagen original. ¿Puede apreciar la reduccion del ruido?

In [ ]:
# ──────────────────────────────────────────────────────────────
# E7 — Promediar 11 imagenes consecutivas (reduccion de ruido sin
#       atenuar altas frecuencias — tecnica clasica en astronomia)
# PDF: "Otra forma de reducir el ruido sin atenuar las altas frecuencias
#       es promediar imagenes consecutivas. Esta tecnica se usa
#       frecuentemente en Astronomia. Promedie 11 imagenes consecutivas
#       de la fibra y compare el resultado con la imagen original.
#       ¿Puede apreciar la reduccion del ruido?"
# Rodrigo: promedio sobre fibra3 (1) a fibra3 (10)  → solo 10 imagenes (deberia ser 11).
# ──────────────────────────────────────────────────────────────
fibras = [f.escala_grises() for f in load_fibras(11)]
fibras[0].title = "fibra3 (1) original"
fibras[0].mostrar(block=False)
fibras[0].histograma(block=False)

# Approach: usar el metodo nuevo de la lib promediar(otros).
# Teoria: si cada toma tiene ruido r_i ~ N(0, σ²) independiente, el
# promedio de N tomas tiene ruido ~ N(0, σ²/N). La desviacion estandar
# del ruido baja por factor √N. Para N=11: √11 ≈ 3.3 → ruido ~3x menor.
# La señal real se mantiene intacta (no es un filtro pasa-bajas, no atenua frecuencias).
#
# TODO:
#   promedio = fibras[0].promediar(fibras[1:])    # ya estan los 11
#   promedio.title = "Promedio de 11 fibras"
#   promedio.mostrar(block=False)
#   promedio.histograma(block=False)
#
# TODO (comparacion lado a lado):
#   fibras[0].mostrar_diferencias(promedio, magnifier=10.0, block=False)
#   # magnifier=10 hace ver donde se "limpio" el ruido.
#
# TODO (cuantitativo — confirmar la teoria √N):
#   ruido_original = (fibras[0].tensor - fibras[1].tensor).std().item()
#   ruido_promedio = (promedio.tensor - fibras[5].tensor).std().item()
#   reduccion = ruido_original / ruido_promedio
#   print(f"σ_original={ruido_original:.4f}, σ_promedio={ruido_promedio:.4f}")
#   print(f"reduccion ≈ {reduccion:.2f}x  (teorica √11 ≈ 3.32x)")

plt.show()

### Explicaciones del Ejercicio 7

**Descripción:** _TODO — qué imágenes se promediaron (11 de la fibra), qué método de la lib se usó (`promediar`), qué se midió cuantitativamente._

**¿Se aprecia la reducción del ruido? (justificar):**

_TODO_

Pista: **sí**, y se puede demostrar con números. El fundamento teórico:

> Si cada toma `f_i` tiene la misma señal real `s` más ruido independiente `r_i ~ N(0, σ²)`, entonces:
>
> $$\bar{f} = \frac{1}{N} \sum f_i = s + \frac{1}{N} \sum r_i$$
>
> El promedio de `N` variables normales independientes tiene varianza `σ²/N`, así que `std → σ/√N`.

Para **N = 11** la reducción esperada es `√11 ≈ 3.32×` menos ruido. Comparado con un filtro pasa-bajas (gaussiano, mediana), esta técnica **no destruye los detalles finos** porque solo opera sobre el eje temporal — la señal espacial queda intacta. Por eso es la técnica preferida en astronomía (stacking) y microscopía de baja luz.

## Conclusiones

_TODO: cuando son utiles y cuando no funcionan bien las tecnicas usadas — pasa-bajas, mediana, gaussiano, High-Boost, deteccion de bordes, resta y promedio de imagenes._